# 🧠 Preferential Mixture-of-Experts
### Replication of Pradier et al., 2021 — Pima Indians Diabetes Dataset

---
## 📁 Cell 1 — Upload & Inspect Data

In [ ]:
# from google.colab import files
# import io, pandas as pd

# print("👇 Click 'Choose Files' below and select your diabetes.csv")
# uploaded = files.upload()

# Load whichever file was uploaded
# filename = list(uploaded.keys())[0]
# df_check = pd.read_csv(io.BytesIO(uploaded[filename]))

# Save to disk so the rest of the notebook can use it
#df_check.to_csv('diabetes.csv', index=False)

# print(f"\n✅ Uploaded '{filename}' successfully!")
# print(f"   Shape: {df_check.shape[0]} rows × {df_check.shape[1]} columns")
# print(f"   Columns: {list(df_check.columns)}")
# df_check.head()

import pandas as pd
import os

# Define the path to your data folder
data_path = os.path.join('data', 'diabetes.csv')

if os.path.exists(data_path):
    df_check = pd.read_csv(data_path)
    print(f"✅ Data loaded from {data_path}")
else:
    print(f"❌ Error: Could not find {data_path}. Please ensure the 'data' folder exists.")

df_check.head()

---
## 📦 Cell 2 — Install & Import Libraries

In [ ]:
# All libraries below are pre-installed in Colab
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.special import expit
from scipy.optimize import minimize
from numpy.linalg import svd as _svd
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('✅ All libraries imported successfully!')

---
## 🗂️ Cell 3 — Load Data & Define Human Rules

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────
df = pd.read_csv('diabetes.csv')
X_raw = df.drop('Outcome', axis=1).values.astype(float)
y     = df['Outcome'].values.astype(float)

feature_names = df.drop('Outcome', axis=1).columns.tolist()
GLUCOSE_IDX   = feature_names.index('Glucose')
BMI_IDX       = feature_names.index('BMI')
AGE_IDX       = feature_names.index('Age')

print(f'✅ Data loaded: {X_raw.shape[0]} samples, {X_raw.shape[1]} features')
print(f'   Features: {feature_names}')
print(f'   Class balance — 0: {(y==0).sum()}  |  1: {(y==1).sum()}')

# ── Human rule function g(x) ───────────────────────────────────────────
# Returns:  1.0  → diabetic
#           0.0  → non-diabetic
#          -1.0  → rule not applicable
def human_rule(X_raw):
    g       = np.full(len(X_raw), -1.0)
    glucose = X_raw[:, GLUCOSE_IDX]
    bmi     = X_raw[:, BMI_IDX]
    age     = X_raw[:, AGE_IDX]
    valid   = (glucose > 0) & (bmi > 0) & (age > 0)

    # --- Positive (diabetic) rules ---
    # Standard WHO/ADA diagnostic threshold
    g[valid & (glucose >= 126)]                          = 1.0
    # Obese + elevated glucose (prediabetic range)
    g[valid & (bmi >= 30) & (glucose >= 110)]            = 1.0
    # Older + overweight + raised glucose
    g[valid & (age >= 45) & (bmi >= 25) & (glucose >= 110)] = 1.0

    # --- Negative (non-diabetic) rules ---
    # Standard clinical "normal" glucose
    g[valid & (glucose < 100) & (bmi < 25)]              = 0.0
    # Young + healthy BMI
    g[valid & (age < 30) & (bmi < 22)]                   = 0.0
    # Normal glucose regardless of other factors
    g[valid & (glucose < 100) & (age < 45) & (bmi < 30)] = 0.0

    return g

g_check = human_rule(X_raw)
print(f'\n   Human rule coverage: {(g_check != -1).sum()} / {len(g_check)} samples '
      f'({100*(g_check != -1).mean():.1f}%)')

---
## ⚙️ Cell 4 — Helper Functions & Model Definitions

In [ ]:
# ── Core helpers ───────────────────────────────────────────────────────
def sigmoid(z):
    return expit(np.clip(z, -30, 30))

def log_loss_reg(theta, X, y, gamma=0.0):
    p   = np.clip(sigmoid(X @ theta), 1e-7, 1-1e-7)
    nll = -np.mean(y * np.log(p) + (1-y) * np.log(1-p))
    return nll + gamma * np.sum(np.abs(theta))

def grad_log_loss_reg(theta, X, y, gamma=0.0):
    p    = sigmoid(X @ theta)
    grad = X.T @ (p - y) / len(y)
    grad += gamma * np.sign(theta)
    return grad

def predict_proba(theta, X):
    return sigmoid(X @ theta)

def add_bias(X):
    return np.hstack([np.ones((len(X), 1)), X])

def moe_predict(theta, w, X_raw, g, bias_X):
    rho   = sigmoid(bias_X @ w)
    f     = predict_proba(theta, bias_X)
    known = (g != -1)
    y_hat = f.copy()
    y_hat[known] = (1-rho[known])*f[known] + rho[known]*g[known]
    return np.clip(y_hat, 1e-7, 1-1e-7)

def moe_loss(theta, w, X_raw, y, g, bias_X, gamma=0.01):
    y_hat = moe_predict(theta, w, X_raw, g, bias_X)
    nll   = -np.mean(y*np.log(y_hat) + (1-y)*np.log(1-y_hat))
    return nll + gamma * np.sum(np.abs(w))

def brier_score(y_true, y_prob):
    return np.mean((y_prob - y_true) ** 2)

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_acc  = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(bin_acc - bin_conf)
    return ece

# ── KL regularisation ─────────────────────────────────────────────────
def bernoulli_kl(p, q, eps=1e-7):
    p = np.clip(p, eps, 1-eps)
    q = np.clip(q, eps, 1-eps)
    return p*np.log(p/q) + (1-p)*np.log((1-p)/(1-q))

def moe_loss_kl(theta, w, X_raw, y, g, bias_X, gamma=0.01, beta=2.0, mu=0.1):
    y_hat = moe_predict(theta, w, X_raw, g, bias_X)
    nll   = -np.mean(y*np.log(y_hat) + (1-y)*np.log(1-y_hat))
    reg   = gamma * np.sum(np.abs(w))
    known = (g != -1)
    if known.sum() == 0:
        return nll + reg
    kl       = bernoulli_kl(g[known], y_hat[known])
    rho      = np.clip(sigmoid(bias_X[known] @ w), 1e-7, 1-1e-7)
    cov_push = -mu * float(np.mean(np.log(rho)))
    return nll + reg + beta * float(np.mean(kl)) + cov_push

# ── Coverage metrics ──────────────────────────────────────────────────
def soft_coverage(w, bias_X, g):
    known = g != -1
    if known.sum() == 0:
        return 0.0
    return float(np.mean(sigmoid(bias_X[known] @ w))) * 100.0

def hard_coverage_curve(theta, w, bias_X, g, y, thresholds):
    p     = predict_proba(theta, bias_X)
    rho   = sigmoid(bias_X @ w)
    known = g != -1
    curve = []
    for thr in thresholds:
        use_h  = known & (rho >= thr)
        preds  = np.where(use_h, g, (p >= 0.5).astype(float))
        acc    = np.mean(preds == y) * 100
        cov    = np.mean(use_h) * 100
        curve.append((cov, acc))
    return curve

print('✅ Helper functions defined!')

# ── Model trainers ────────────────────────────────────────────────────
def train_standard_moe(X_raw, y, g, bias_X, gamma=0.01, n_restarts=3):
    d = bias_X.shape[1]
    best_loss, best_theta, best_w = np.inf, None, None
    for _ in range(n_restarts):
        theta0 = np.random.randn(d) * 0.01
        w0     = np.random.randn(d) * 0.01
        def joint_loss(params):
            return moe_loss(params[:d], params[d:], X_raw, y, g, bias_X, gamma)
        res = minimize(joint_loss, np.concatenate([theta0, w0]),
                       method='L-BFGS-B', options={'maxiter': 500, 'ftol': 1e-9})
        if res.fun < best_loss:
            best_loss  = res.fun
            best_theta = res.x[:d]
            best_w     = res.x[d:]
    return best_theta, best_w, best_loss

def train_preferential_moe(X_raw, y, g, bias_X,
                            theta_star, w_star, L_star,
                            gamma=0.01, eps=0.1, t=3.0, n_restarts=3):
    d      = bias_X.shape[1]
    budget = (1 + eps) * L_star
    def lb_loss(params):
        theta, w = params[:d], params[d:]
        rho      = np.clip(sigmoid(bias_X @ w), 1e-7, 1-1e-7)
        L_cur    = moe_loss(theta, w, X_raw, y, g, bias_X, gamma)
        barrier  = budget - L_cur
        if barrier <= 0:
            return 1e10
        return -t * np.sum(np.log(rho)) - np.log(barrier)
    best_val, best_theta, best_w = np.inf, None, None
    for _ in range(n_restarts):
        theta0 = theta_star + np.random.randn(d) * 0.01
        w0     = w_star     + np.random.randn(d) * 0.01
        res    = minimize(lb_loss, np.concatenate([theta0, w0]),
                          method='L-BFGS-B', options={'maxiter': 1000, 'ftol': 1e-10})
        th_c, w_c = res.x[:d], res.x[d:]
        L_c = moe_loss(th_c, w_c, X_raw, y, g, bias_X, gamma)
        if L_c <= budget + 1e-5 and res.fun < best_val:
            best_val, best_theta, best_w = res.fun, th_c, w_c
    if best_theta is None:
        best_theta, best_w = theta_star.copy(), w_star.copy()
    return best_theta, best_w

def train_preferential_moe_kl(X_raw, y, g, bias_X,
                              theta_star, w_star,
                              gamma=0.01, beta=2.0, mu=0.1, n_restarts=3):
    d = bias_X.shape[1]
    best_loss, best_theta, best_w = np.inf, None, None
    for _ in range(n_restarts):
        theta0 = theta_star + np.random.randn(d) * 0.01
        w0     = w_star     + np.random.randn(d) * 0.01
        def joint_loss(params):
            theta, w = params[:d], params[d:]
            return moe_loss_kl(theta, w, X_raw, y, g, bias_X, gamma=gamma, beta=beta, mu=mu)
        res = minimize(joint_loss, np.concatenate([theta0, w0]),
                       method='L-BFGS-B', options={'maxiter': 1000, 'ftol': 1e-10})
        if res.fun < best_loss:
            best_loss  = res.fun
            best_theta = res.x[:d]
            best_w     = res.x[d:]
    if best_theta is None:
        best_theta, best_w = theta_star.copy(), w_star.copy()
    return best_theta, best_w

print('✅ Model trainers defined!')

# ── Conformal Abstainer ───────────────────────────────────────────────
class ConformalAbstainer:
    """Split-conformal OOD detector using Mahalanobis distance."""
    def __init__(self, alpha=0.02):
        self.alpha   = alpha
        self.tau     = None
        self.mu      = None
        self.cov_inv = None

    def _mahal(self, X):
        diff = X - self.mu
        return np.sqrt(np.einsum('ij,jk,ik->i', diff, self.cov_inv, diff))

    def fit(self, X_cal):
        self.mu  = X_cal.mean(axis=0)
        cov      = np.cov(X_cal, rowvar=False)
        cov     += 1e-6 * np.eye(cov.shape[0])
        self.cov_inv = np.linalg.inv(cov)
        scores   = self._mahal(X_cal)
        n        = len(scores)
        q_level  = np.ceil((1 - self.alpha) * (n + 1)) / n
        self.tau = np.quantile(scores, min(q_level, 1.0))
        return self

    def is_ood(self, X_test):
        return self._mahal(X_test) > self.tau

    def predict(self, X_test, y_hat_moe, g_test):
        ood       = self.is_ood(X_test)
        final     = y_hat_moe.copy()
        abstained = np.zeros(len(X_test), dtype=bool)
        for i in np.where(ood)[0]:
            if g_test[i] != -1:
                final[i] = float(g_test[i])
            else:
                final[i]     = 0.5
                abstained[i] = True
        return np.clip(final, 1e-7, 1-1e-7), abstained

print('✅ Conformal Abstainer defined!')

# ── Conformal Prediction on Gating Function ─────────────────────────────
class ConformalGating:
    """
    Split conformal prediction on the gating function.

    The gating function is treated as a binary classifier:
      - class 1 = "follow human rule"
      - class 0 = "use ML"

    True gating labels for calibration are derived from which expert
    was actually correct:
      - If human rule correct              -> true label = 1 (follow human)
      - If human rule wrong and ML correct -> true label = 0 (use ML)
      - If both correct                    -> true label = 1 (preferential philosophy)
      - If both wrong                      -> true label = 1 (no reason to override)

    The nonconformity score is s = 1 - p_hat(true gating label).
    """

    def __init__(self, alpha=0.05):
        self.alpha = alpha
        self.threshold = None

    def _derive_gating_labels(self, g_cal, y_cal, ml_preds_cal):
        """
        Derive the 'true' gating label for each calibration patient.

        Returns
        -------
        gating_labels : array, shape (n,)
            1 = "follow human rule", 0 = "use ML", -1 = "not applicable"
        """
        gating_labels = np.full(len(g_cal), -1.0)
        for i in range(len(g_cal)):
            if g_cal[i] == -1:
                gating_labels[i] = -1
                continue
            human_correct = (g_cal[i] == y_cal[i])
            ml_correct = (ml_preds_cal[i] == y_cal[i])
            if human_correct:
                gating_labels[i] = 1.0
            elif ml_correct:
                gating_labels[i] = 0.0
            else:
                gating_labels[i] = 1.0
        return gating_labels

    def calibrate(self, w, bias_X_cal, g_cal, y_cal, ml_preds_cal):
        """
        Compute nonconformity scores on the calibration set and find threshold.
        """
        rho = np.clip(sigmoid(bias_X_cal @ w), 1e-7, 1 - 1e-7)
        gating_labels = self._derive_gating_labels(g_cal, y_cal, ml_preds_cal)

        valid = gating_labels != -1
        if valid.sum() == 0:
            self.threshold = 1.0
            return self

        rho_valid = rho[valid]
        labels_valid = gating_labels[valid]

        # s = 1 - p_hat(true gating label)
        scores = np.where(
            labels_valid == 1,
            1 - rho_valid,    # true = "follow human", confidence = rho
            rho_valid          # true = "use ML", confidence = 1 - rho
        )

        self.scores = scores
        n = len(scores)
        q_level = np.ceil((1 - self.alpha) * (n + 1)) / n
        self.threshold = np.quantile(scores, min(q_level, 1.0))
        return self

    def predict_sets(self, w, bias_X_test, g_test):
        """
        Produce conformal prediction sets for the gating decision.

        Returns list of sets: {1}=follow human, {0}=use ML, {0,1}=uncertain
        """
        rho = np.clip(sigmoid(bias_X_test @ w), 1e-7, 1 - 1e-7)
        gating_sets = []
        for i in range(len(g_test)):
            if g_test[i] == -1:
                gating_sets.append({0})
                continue
            prediction_set = set()
            if (1 - rho[i]) <= self.threshold:
                prediction_set.add(1)
            if rho[i] <= self.threshold:
                prediction_set.add(0)
            if len(prediction_set) == 0:
                if rho[i] >= 0.5:
                  prediction_set = {1}
                else:
                  prediction_set = {0}
            gating_sets.append(prediction_set)
        return gating_sets

print('✅ Conformal Gating defined!')

# ── Conformal Prediction on ML Classifier ─────────────────────────────
class ConformalMLClassifier:
    """
    Split conformal prediction on the ML classifier.
    Nonconformity score: s = 1 - p_hat(true label).
    """

    def __init__(self, alpha=0.05):
        self.alpha = alpha
        self.threshold = None

    def calibrate(self, theta, bias_X_cal, y_cal):
        probs = np.clip(sigmoid(bias_X_cal @ theta), 1e-7, 1 - 1e-7)
        scores = np.where(y_cal == 1, 1 - probs, probs)
        self.scores = scores
        n = len(scores)
        q_level = np.ceil((1 - self.alpha) * (n + 1)) / n
        self.threshold = np.quantile(scores, min(q_level, 1.0))
        return self

    def predict_sets(self, theta, bias_X_test):
        """Returns list of sets: {1}=positive, {0}=negative, {0,1}=uncertain"""
        probs = np.clip(sigmoid(bias_X_test @ theta), 1e-7, 1 - 1e-7)
        ml_sets = []
        for i in range(len(probs)):
            prediction_set = set()
            if (1 - probs[i]) <= self.threshold:
                prediction_set.add(1)
            if probs[i] <= self.threshold:
                prediction_set.add(0)
            if len(prediction_set) == 0:
                if probs[i] >= 0.5:
                    prediction_set = {1}
                else:
                    prediction_set = {0}
            ml_sets.append(prediction_set)
        return ml_sets

print('✅ Conformal ML Classifier defined!')

# ── Combined Deferral System ─────────────────────────────
class ConformalDeferralSystem:
    """
    Combines CP on gating + CP on ML into the four/six-outcome decision system.

    Outcomes:
      "follow_human"   - gating confidently selects human
      "use_ml"         - gating selects ML AND ML confident
      "fallback_human" - ML selected but uncertain; human rule exists
      "escalate"       - gating uncertain, OR ML uncertain + no human rule
      "ood_human"      - OOD patient, human rule exists (if OOD mask provided)
      "ood_abstain"    - OOD patient, no human rule (if OOD mask provided)
    """

    def __init__(self, alpha_gating=0.05, alpha_ml=0.05,
             cp_gating_on=True, cp_ml_on=True):
        self.cp_gating_on = cp_gating_on
        self.cp_ml_on = cp_ml_on
        self.cp_gating = ConformalGating(alpha=alpha_gating) if cp_gating_on else None
        self.cp_ml = ConformalMLClassifier(alpha=alpha_ml) if cp_ml_on else None

    def calibrate(self, theta, w, bias_X_cal, g_cal, y_cal):
        ml_probs = sigmoid(bias_X_cal @ theta)
        ml_preds = (ml_probs >= 0.5).astype(float)
        if self.cp_gating is not None:
            self.cp_gating.calibrate(w, bias_X_cal, g_cal, y_cal, ml_preds)
        if self.cp_ml is not None:
            self.cp_ml.calibrate(theta, bias_X_cal, y_cal)
        return self

    def predict(self, theta, w, bias_X_test, g_test, ood_mask=None):
        n = len(g_test)
        outcomes = np.array(["" for _ in range(n)], dtype=object)
        predictions = np.full(n, np.nan)
        ml_probs = np.clip(sigmoid(bias_X_test @ theta), 1e-7, 1 - 1e-7)
        rho = sigmoid(bias_X_test @ w)

        # Step 0: OOD check (optional)
        if ood_mask is not None:
            for i in np.where(ood_mask)[0]:
                if g_test[i] != -1:
                    outcomes[i] = "ood_human"
                    predictions[i] = g_test[i]
                else:
                    outcomes[i] = "ood_abstain"
                    predictions[i] = 0.5

        remaining_idx = np.where(outcomes == "")[0]
        if len(remaining_idx) == 0:
            return outcomes, predictions, self._summary(outcomes), ml_sets, gating_sets

        # Pre-compute prediction sets only for active components
        gating_sets = None
        ml_sets = None
        if self.cp_gating is not None:
            gating_sets = self.cp_gating.predict_sets(w, bias_X_test, g_test)
        if self.cp_ml is not None:
            ml_sets = self.cp_ml.predict_sets(theta, bias_X_test)

        for i in remaining_idx:

            # -- GATING DECISION --
            if g_test[i] == -1:
                gating_decision = "use_ml"
            elif self.cp_gating_on and gating_sets is not None:
                g_set = gating_sets[i]
                if g_set == {1}:
                    gating_decision = "follow_human"
                elif g_set == {0, 1}:
                    gating_decision = "escalate"
                else:
                    gating_decision = "use_ml"
            else:
                # CP gating OFF -> original threshold routing
                if rho[i] >= 0.5:
                    gating_decision = "follow_human"
                else:
                    gating_decision = "use_ml"

            # -- ACT ON GATING DECISION --
            if gating_decision == "follow_human":
                outcomes[i] = "follow_human"
                predictions[i] = g_test[i]
                continue
            if gating_decision == "escalate":
                outcomes[i] = "escalate"
                predictions[i] = 0.5
                continue

            # -- ML DECISION --
            if self.cp_ml_on and ml_sets is not None:
                m_set = ml_sets[i]
                if len(m_set) == 1:
                    outcomes[i] = "use_ml"
                    predictions[i] = ml_probs[i]
                else:
                    if g_test[i] != -1:
                        outcomes[i] = "fallback_human"
                        predictions[i] = g_test[i]
                    else:
                        outcomes[i] = "escalate"
                        predictions[i] = 0.5
            else:
                # CP ML OFF -> use ML prediction directly
                outcomes[i] = "use_ml"
                predictions[i] = ml_probs[i]

        return outcomes, predictions, self._summary(outcomes), ml_sets, gating_sets

    def _summary(self, outcomes):
        unique, counts = np.unique(outcomes, return_counts=True)
        summary = dict(zip(unique, counts))
        total = len(outcomes)
        return {
            "counts": summary,
            "rates": {k: v / total * 100 for k, v in summary.items()},
            "total": total
        }

# ── Evaluation helpers ────────────────────────────────────────────────
def compute_ml_accuracy(outcomes, predictions, y_true):
    """
    Among patients routed to ML (outcome == 'use_ml'),
    what fraction were predicted correctly (hard threshold 0.5).
    """
    ml_mask = (outcomes == "use_ml")
    if ml_mask.sum() == 0:
        return np.nan
    ml_preds_hard = (predictions[ml_mask] >= 0.5).astype(float)
    return (ml_preds_hard == y_true[ml_mask]).mean()

def compute_conformal_coverage(ml_sets, outcomes, y_true, gating_sets=None):
    """
    Empirical conformal coverage.

    For CP-ML: among use_ml and fallback_human patients, fraction
    where y_true is inside the ML prediction set.

    For CP-gating: fraction of patients NOT escalated, directly
    reflecting the 1-alpha guarantee.
    """
    if ml_sets is not None:
        mask = np.isin(outcomes, ['use_ml', 'fallback_human'])
        if mask.sum() == 0:
            return np.nan
        covered = sum(
            y_true[i] in ml_sets[i]
            for i in np.where(mask)[0]
        )
        return covered / mask.sum()

    if gating_sets is not None:
        # For CP-gating, coverage = fraction of patients NOT escalated
        mask = np.isin(outcomes, ['follow_human', 'use_ml', 'fallback_human'])
        return mask.sum() / len(outcomes)

    return np.nan

def compute_outcome_accuracy(outcomes, predictions, y_true):
    """Compute accuracy within each outcome category."""
    results = {}
    for outcome in ["follow_human", "use_ml", "fallback_human",
                    "ood_human", "ood_abstain", "escalate"]:
        mask = (outcomes == outcome)
        if mask.sum() == 0:
            results[outcome] = {"n": 0, "accuracy": np.nan}
            continue
        if outcome in ["escalate", "ood_abstain"]:
            results[outcome] = {"n": int(mask.sum()), "accuracy": np.nan}
        else:
            preds_hard = (predictions[mask] >= 0.5).astype(float)
            acc = (preds_hard == y_true[mask]).mean()
            results[outcome] = {"n": int(mask.sum()), "accuracy": acc}
    return results

def record_cp_model(key, outcomes, predictions, ml_sets, y_test, w, bias_X_test, g_test, summary, gating_sets=None):
    pred_filled = np.where(np.isnan(predictions), 0.5, predictions)
    auc_all  = roc_auc_score(y_test, pred_filled)
    non_esc  = ~np.isin(outcomes, ["escalate", "ood_abstain"])
    auc_ne   = (roc_auc_score(y_test[non_esc], predictions[non_esc])
                if (non_esc.sum() > 0 and len(np.unique(y_test[non_esc])) > 1) else np.nan)
    esc_rate = summary["rates"].get("escalate", 0.0) + summary["rates"].get("ood_abstain", 0.0)
    ml_acc   = compute_ml_accuracy(outcomes, predictions, y_test)
    cp_cov   = compute_conformal_coverage(ml_sets, outcomes, y_test, gating_sets=gating_sets)
    bs_all   = brier_score(y_test, pred_filled)
    bs_ne    = (brier_score(y_test[non_esc], predictions[non_esc])
                if non_esc.sum() > 0 else np.nan)
    ece_all = expected_calibration_error(y_test, pred_filled)
    ece_ne  = (expected_calibration_error(y_test[non_esc], predictions[non_esc])
           if non_esc.sum() > 0 else np.nan)

    records[key]["auc"].append(auc_all)
    records[key]["auc_non_abstain"].append(auc_ne)
    records[key]["cov"].append(soft_coverage(w, bias_X_test, g_test))
    records[key]["eff_cov"].append(np.mean(outcomes == "follow_human") * 100)
    records[key]["abstain_rate"].append(summary["rates"].get("ood_abstain", 0.0))
    records[key]["escalation_rate"].append(esc_rate)
    records[key]["ml_accuracy"].append(ml_acc)
    records[key]["conformal_coverage"].append(cp_cov)
    records[key]["brier"].append(bs_all)
    records[key]["brier_non_esc"].append(bs_ne)
    for outcome, count in summary["counts"].items():
        outcome_counts_all[key][outcome] = outcome_counts_all[key].get(outcome, 0) + count
    records[key]["ece"].append(ece_all)
    records[key]["ece_non_esc"].append(ece_ne)

print('✅ Conformal Deferral System defined!')


---
## 🔁 Cell 5a — Setup & Baseline Loop (CV Training)

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────
N_SEEDS, N_FOLDS = 5, 5
GAMMA, EPS, T  = 0.01, 0.02, 0.1
BETA_KL        = 2.0
MU_COV         = 0.1
ALPHA_CONF_OOD = 0.002   # alpha for OOD detection
ALPHA_CP       = 0.10    # alpha for CP on model outputs
THRESHOLDS     = np.linspace(0.0, 1.0, 40)
CAL_FRACTION   = 0.20    # fraction of training fold held out for calibration

# ── Model keys ──
KEYS_BASE = ["ml_only", "std_moe", "pref_moe", "pref_moe_kl", "conf_kl"]
KEYS_LB   = ["lb_cp_ml", "lb_cp_gating", "lb_cp_both"]
KEYS_KL   = ["kl_cp_ml", "kl_cp_gating", "kl_cp_both"]
KEYS      = KEYS_BASE + KEYS_LB + KEYS_KL

records = {k: {"auc": [], "cov": [], "eff_cov": [], "auc_non_abstain": [],
               "abstain_rate": [], "escalation_rate": [],
               "ml_accuracy": [], "conformal_coverage": [],
               "brier": [], "brier_non_esc": [],
               "ece": [], "ece_non_esc": []}
           for k in KEYS}
ac_curves = {k: [] for k in KEYS if k != "ml_only"}
stored_weights = {}

KEYS_WITH_OUTCOMES = KEYS_LB + KEYS_KL

outcome_counts_all = {k: {} for k in KEYS_WITH_OUTCOMES}

# Abstained sample breakdown (for OOD model, same as before)
abstain_breakdown = {"correct_rule": [], "wrong_rule": [],
                     "correct_ml":   [], "wrong_ml":   []}

print(f'Running {N_SEEDS}-seed × {N_FOLDS}-fold cross-validation …')
print('(Each dot = 1 fold completed)\n')

for seed in range(N_SEEDS):
    np.random.seed(seed)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_raw, y)):
        try:
            rs = seed * N_FOLDS + fold

            # ── Split training fold into train-proper + calibration ──────
            train_proper_idx, cal_idx = train_test_split(
                tr_idx, test_size=CAL_FRACTION, stratify=y[tr_idx], random_state=rs
            )

            # Raw features
            Xtr_r  = X_raw[train_proper_idx]
            Xcal_r = X_raw[cal_idx]
            Xte_r  = X_raw[te_idx]
            ytr    = y[train_proper_idx]
            ycal   = y[cal_idx]
            yte    = y[te_idx]
            gtr    = human_rule(Xtr_r)
            gcal   = human_rule(Xcal_r)
            gte    = human_rule(Xte_r)

            # Scale using train-proper statistics only
            sc = StandardScaler()
            Xtr  = sc.fit_transform(Xtr_r)
            Xcal = sc.transform(Xcal_r)
            Xte  = sc.transform(Xte_r)
            bXtr  = add_bias(Xtr)
            bXcal = add_bias(Xcal)
            bXte  = add_bias(Xte)
            d = bXtr.shape[1]

            # ── Model Training ───────────────────────────────────────

            # ── 1. ML Only (Baseline) ──────────────────────────────────
            res = minimize(lambda th: log_loss_reg(th, bXtr, ytr, GAMMA),
                          np.zeros(d),
                          jac=lambda th: grad_log_loss_reg(th, bXtr, ytr, GAMMA),
                          method="L-BFGS-B", options={"maxiter": 500})
            th_ml = res.x
            p_ml  = predict_proba(th_ml, bXte)
            records["ml_only"]["auc"].append(roc_auc_score(yte, p_ml))
            records["ml_only"]["auc_non_abstain"].append(roc_auc_score(yte, p_ml))
            records["ml_only"]["cov"].append(0.0)
            records["ml_only"]["eff_cov"].append(np.nan)
            records["ml_only"]["abstain_rate"].append(0.0)
            records["ml_only"]["escalation_rate"].append(0.0)
            records["ml_only"]["ml_accuracy"].append(np.nan)
            records["ml_only"]["conformal_coverage"].append(np.nan)
            records["ml_only"]["brier"].append(brier_score(yte, p_ml))
            records["ml_only"]["brier_non_esc"].append(brier_score(yte, p_ml))
            records["ml_only"]["ece"].append(expected_calibration_error(yte, p_ml))
            records["ml_only"]["ece_non_esc"].append(expected_calibration_error(yte, p_ml))

            # ── 2. Standard MoE ────────────────────────────────────────
            th_s, w_s, L_star = train_standard_moe(Xtr_r, ytr, gtr, bXtr, GAMMA)
            p_std = moe_predict(th_s, w_s, Xte_r, gte, bXte)
            records["std_moe"]["auc"].append(roc_auc_score(yte, p_std))
            records["std_moe"]["auc_non_abstain"].append(roc_auc_score(yte, p_std))
            records["std_moe"]["cov"].append(soft_coverage(w_s, bXte, gte))
            records["std_moe"]["eff_cov"].append(np.nan)
            records["std_moe"]["abstain_rate"].append(0.0)
            records["std_moe"]["escalation_rate"].append(0.0)
            records["std_moe"]["ml_accuracy"].append(np.nan)
            records["std_moe"]["conformal_coverage"].append(np.nan)
            ac_curves["std_moe"].append(
                hard_coverage_curve(th_s, w_s, bXte, gte, yte, THRESHOLDS))
            records["std_moe"]["brier"].append(brier_score(yte, p_std))
            records["std_moe"]["brier_non_esc"].append(brier_score(yte, p_std))
            records["std_moe"]["ece"].append(expected_calibration_error(yte, p_std))
            records["std_moe"]["ece_non_esc"].append(expected_calibration_error(yte, p_std))

            # ── 3. Preferential MoE (log-barrier) ──────────────────────
            th_p, w_p = train_preferential_moe(
                Xtr_r, ytr, gtr, bXtr, th_s, w_s, L_star, GAMMA, EPS, T)
            p_pref = moe_predict(th_p, w_p, Xte_r, gte, bXte)
            records["pref_moe"]["auc"].append(roc_auc_score(yte, p_pref))
            records["pref_moe"]["auc_non_abstain"].append(roc_auc_score(yte, p_pref))
            records["pref_moe"]["cov"].append(soft_coverage(w_p, bXte, gte))
            records["pref_moe"]["eff_cov"].append(np.nan)
            records["pref_moe"]["abstain_rate"].append(0.0)
            records["pref_moe"]["escalation_rate"].append(0.0)
            records["pref_moe"]["ml_accuracy"].append(np.nan)
            records["pref_moe"]["conformal_coverage"].append(np.nan)
            ac_curves["pref_moe"].append(
                hard_coverage_curve(th_p, w_p, bXte, gte, yte, THRESHOLDS))
            records["pref_moe"]["brier"].append(brier_score(yte, p_pref))
            records["pref_moe"]["brier_non_esc"].append(brier_score(yte, p_pref))
            records["pref_moe"]["ece"].append(expected_calibration_error(yte, p_pref))
            records["pref_moe"]["ece_non_esc"].append(expected_calibration_error(yte, p_pref))

            # ── 4. Preferential MoE (KL-regularised) ──────────────────
            np.random.seed(seed * 1000 + fold)
            th_k, w_k = train_preferential_moe_kl(
                Xtr_r, ytr, gtr, bXtr, th_s, w_s, gamma=GAMMA, beta=BETA_KL, mu=MU_COV)
            p_kl = moe_predict(th_k, w_k, Xte_r, gte, bXte)
            records["pref_moe_kl"]["auc"].append(roc_auc_score(yte, p_kl))
            records["pref_moe_kl"]["auc_non_abstain"].append(roc_auc_score(yte, p_kl))
            records["pref_moe_kl"]["cov"].append(soft_coverage(w_k, bXte, gte))
            records["pref_moe_kl"]["eff_cov"].append(np.nan)
            records["pref_moe_kl"]["abstain_rate"].append(0.0)
            records["pref_moe_kl"]["escalation_rate"].append(0.0)
            records["pref_moe_kl"]["ml_accuracy"].append(np.nan)
            records["pref_moe_kl"]["conformal_coverage"].append(np.nan)
            ac_curves["pref_moe_kl"].append(
                hard_coverage_curve(th_k, w_k, bXte, gte, yte, THRESHOLDS))
            records["pref_moe_kl"]["brier"].append(brier_score(yte, p_kl))
            records["pref_moe_kl"]["brier_non_esc"].append(brier_score(yte, p_kl))
            records["pref_moe_kl"]["ece"].append(expected_calibration_error(yte, p_kl))
            records["pref_moe_kl"]["ece_non_esc"].append(expected_calibration_error(yte, p_kl))

            # ── Abstain breakdown (uses p_kl computed above) ──────────
            abstainer_temp = ConformalAbstainer(alpha=ALPHA_CONF_OOD).fit(Xcal)
            _, abstained_temp = abstainer_temp.predict(Xte, p_kl, gte)
            for i in np.where(abstained_temp)[0]:
                kl_pred  = int(p_kl[i] >= 0.5)
                correct  = (kl_pred == int(yte[i]))
                has_rule = (gte[i] != -1)
                if has_rule and correct:
                    abstain_breakdown["correct_rule"].append(1)
                elif has_rule and not correct:
                    abstain_breakdown["wrong_rule"].append(1)
                elif not has_rule and correct:
                    abstain_breakdown["correct_ml"].append(1)
                else:
                    abstain_breakdown["wrong_ml"].append(1)

            # Store weights for LB & KL ablations
            stored_weights[(seed, fold)] = {
                "th_ml": th_ml.copy(), "th_s": th_s.copy(),
                "w_s":   w_s.copy(),   "th_p": th_p.copy(),
                "w_p":   w_p.copy(),   "th_k": th_k.copy(),
                "w_k":   w_k.copy(),
                "bXcal": bXcal.copy(), "bXte": bXte.copy(),
                "gcal":  gcal.copy(),  "gte":  gte.copy(),
                "ycal":  ycal.copy(),  "yte":  yte.copy(),
                "Xcal":  Xcal.copy(),  "Xte":  Xte.copy(),
            }
            print(".", end="", flush=True)

        except Exception as e:
            print(f"\nERROR seed={seed} fold={fold}: {e}")
            import traceback; traceback.print_exc()

print("\n\n✅ Baseline CV complete!")

---
## 🔁 Cell 5b — Log-Barrier Ablations (CV Training)

In [ ]:
for k in KEYS_LB:
    records[k] = {"auc": [], "cov": [], "eff_cov": [], "auc_non_abstain": [],
                  "abstain_rate": [], "escalation_rate": [],
                  "ml_accuracy": [], "conformal_coverage": [],
                  "brier": [], "brier_non_esc": [],
                  "ece": [], "ece_non_esc": []}
    outcome_counts_all[k] = {}

for (seed, fold), data in stored_weights.items():
    try:
        th_p, w_p = data["th_p"], data["w_p"]
        bXcal, bXte = data["bXcal"], data["bXte"]
        gcal, gte = data["gcal"], data["gte"]
        ycal, yte = data["ycal"], data["yte"]

        # ── Log-Barrier CP Ablation ──────────────────
        # 1. LB + CP on ML only
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=False, cp_ml_on=True)
        cds.calibrate(th_p, w_p, bXcal, gcal, ycal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(th_p, w_p, bXte, gte)
        record_cp_model("lb_cp_ml", out, pred, ml_sets, yte, w_p, bXte, gte, summ, gating_sets=gating_sets)

        # 2. LB + CP on gating only
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=True, cp_ml_on=False)
        cds.calibrate(th_p, w_p, bXcal, gcal, ycal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(th_p, w_p, bXte, gte)
        record_cp_model("lb_cp_gating", out, pred, ml_sets, yte, w_p, bXte, gte, summ, gating_sets=gating_sets)

        # 3. LB + CP on ML & gating
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=True, cp_ml_on=True)
        cds.calibrate(th_p, w_p, bXcal, gcal, ycal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(th_p, w_p, bXte, gte)
        record_cp_model("lb_cp_both", out, pred, ml_sets, yte, w_p, bXte, gte, summ, gating_sets=gating_sets)
        print(".", end="", flush=True)

    except Exception as e:
        print(f"\nERROR {seed},{fold}: {e}")
        import traceback; traceback.print_exc()

print("\n\n✅ LB ablation CV complete!")

---
## 🔁 Cell 5c — KL-Regulatisation Ablations (CV Training)

In [ ]:
for k in KEYS_KL + ["conf_kl"]:
    records[k] = {"auc": [], "cov": [], "eff_cov": [], "auc_non_abstain": [],
                  "abstain_rate": [], "escalation_rate": [],
                  "ml_accuracy": [], "conformal_coverage": [],
                  "brier": [], "brier_non_esc": [],
                  "ece": [], "ece_non_esc": []}
    outcome_counts_all[k] = {}

for (seed, fold), data in stored_weights.items():
    try:
        th_k, w_k = data["th_k"], data["w_k"]
        bXcal, bXte = data["bXcal"], data["bXte"]
        gcal, gte = data["gcal"], data["gte"]
        ycal, yte = data["ycal"], data["yte"]
        Xcal, Xte = data["Xcal"], data["Xte"]

        # ── KL-regularised CP Ablation ──────────────────
        # 1. KL + CP on ML only
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=False, cp_ml_on=True)
        cds.calibrate(th_k, w_k, bXcal, gcal, ycal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(th_k, w_k, bXte, gte)
        record_cp_model("kl_cp_ml", out, pred, ml_sets, yte, w_k, bXte, gte, summ, gating_sets=gating_sets)

        # 2. KL + CP on gating only
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=True, cp_ml_on=False)
        cds.calibrate(th_k, w_k, bXcal, gcal, ycal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(th_k, w_k, bXte, gte)
        record_cp_model("kl_cp_gating", out, pred, ml_sets, yte, w_k, bXte, gte, summ, gating_sets=gating_sets)

        # 3. KL + CP on both
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=True, cp_ml_on=True)
        cds.calibrate(th_k, w_k, bXcal, gcal, ycal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(th_k, w_k, bXte, gte)
        record_cp_model("kl_cp_both", out, pred, ml_sets, yte, w_k, bXte, gte, summ, gating_sets=gating_sets)

        # 4 KL + OOD
        p_kl = moe_predict(th_k, w_k, Xte, gte, bXte)
        abstainer = ConformalAbstainer(alpha=ALPHA_CONF_OOD).fit(Xcal)
        p_kl_conf, abstained = abstainer.predict(Xte, p_kl, gte)

        auc_all = roc_auc_score(yte, p_kl_conf)
        mask_na = ~abstained
        auc_non_abst = (roc_auc_score(yte[mask_na], p_kl_conf[mask_na])
                        if (mask_na.sum() > 0 and len(np.unique(yte[mask_na])) > 1)
                        else np.nan)
        records["conf_kl"]["auc"].append(auc_all)
        records["conf_kl"]["auc_non_abstain"].append(auc_non_abst)
        records["conf_kl"]["cov"].append(soft_coverage(w_k, bXte, gte))
        records["conf_kl"]["eff_cov"].append(np.nan)
        records["conf_kl"]["abstain_rate"].append(abstained.mean() * 100.0)
        records["conf_kl"]["escalation_rate"].append(0.0)
        records["conf_kl"]["ml_accuracy"].append(np.nan)
        records["conf_kl"]["conformal_coverage"].append(np.nan)
        ac_curves["conf_kl"].append(
            hard_coverage_curve(th_k, w_k, bXte, gte, yte, THRESHOLDS))
        records["conf_kl"]["brier"].append(brier_score(yte, p_kl_conf))
        records["conf_kl"]["brier_non_esc"].append(
            brier_score(yte[mask_na], p_kl_conf[mask_na]) if mask_na.sum() > 0 else np.nan)
        records["conf_kl"]["ece"].append(expected_calibration_error(yte, p_kl_conf))
        records["conf_kl"]["ece_non_esc"].append(
            expected_calibration_error(yte[mask_na], p_kl_conf[mask_na])
    if mask_na.sum() > 0 else np.nan)
        print(".", end="", flush=True)

    except Exception as e:
        print(f"\nERROR {seed},{fold}: {e}")
        import traceback; traceback.print_exc()

print("\n\n✅ KL ablation CV complete!")

---
## 🔁 Cell 6 - Sensitivity Analysis (CP Alpha)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Sensitivity Analysis: CP-ML alpha sweep
# ════════════════════════════════════════════════════════════════════

ALPHAS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]
sensitivity = {a: {"esc_rate": [], "auc_ne": [], "cp_cov": []} for a in ALPHAS}

for (seed, fold), data in stored_weights.items():
    th_k, w_k = data["th_k"], data["w_k"]
    bXcal, bXte = data["bXcal"], data["bXte"]
    gcal, gte = data["gcal"], data["gte"]
    ycal, yte = data["ycal"], data["yte"]

    for a in ALPHAS:
        cds = ConformalDeferralSystem(alpha_ml=a, cp_gating_on=False, cp_ml_on=True)
        cds.calibrate(th_k, w_k, bXcal, gcal, ycal)

        #if seed == 0 and fold == 0:
            #print(f"  alpha={a:.2f}  threshold={cds.cp_ml.threshold:.4f}  outcomes: {dict(summ['counts'])}")

        out, pred, summ, ml_sets, gating_sets = cds.predict(th_k, w_k, bXte, gte)
        esc = summ["rates"].get("escalate", 0.0)
        cp_cov = compute_conformal_coverage(ml_sets, out, yte) if ml_sets is not None else np.nan
        non_esc = ~np.isin(out, ["escalate", "ood_abstain"])
        auc_ne = (roc_auc_score(yte[non_esc], pred[non_esc])
                  if (non_esc.sum() > 0 and len(np.unique(yte[non_esc])) > 1)
                  else np.nan)
        sensitivity[a]["esc_rate"].append(esc)
        sensitivity[a]["auc_ne"].append(auc_ne)
        sensitivity[a]["cp_cov"].append(cp_cov)

print("\n-- Sensitivity Analysis: KL + CP-ML only across alpha values --")
print(f"  {'alpha':>6}  {'Esc%':>8}  {'AUC(non-esc)':>12}  {'CPCov':>8}  {'Guarantee':>9}")
print(f"  {'-'*6}  {'-'*8}  {'-'*12}  {'-'*8}  {'-'*9}")
for a in ALPHAS:
    em = np.nanmean(sensitivity[a]["esc_rate"])
    am = np.nanmean(sensitivity[a]["auc_ne"])
    cm = np.nanmean(sensitivity[a]["cp_cov"])
    print(f"  {a:>6.2f}  {em:>7.1f}%  {am:>12.4f}  {cm:>8.3f}  {1-a:>9.2f}")

# ════════════════════════════════════════════════════════════════════
# Sensitivity Analysis: CP-Gating alpha sweep
# ════════════════════════════════════════════════════════════════════

sensitivity_gate = {a: {"esc_rate": [], "auc_ne": [], "cp_cov": []} for a in ALPHAS}

for (seed, fold), data in stored_weights.items():
    th_k, w_k = data["th_k"], data["w_k"]
    bXcal, bXte = data["bXcal"], data["bXte"]
    gcal, gte   = data["gcal"], data["gte"]
    ycal, yte   = data["ycal"], data["yte"]

    for a in ALPHAS:
        cds = ConformalDeferralSystem(alpha_gating=a, cp_gating_on=True, cp_ml_on=False)
        cds.calibrate(th_k, w_k, bXcal, gcal, ycal)

        out, pred, summ, ml_sets, gating_sets = cds.predict(th_k, w_k, bXte, gte)
        esc     = summ["rates"].get("escalate", 0.0)
        cp_cov  = compute_conformal_coverage(gating_sets, out, yte) if gating_sets is not None else np.nan
        non_esc = ~np.isin(out, ["escalate", "ood_abstain"])
        auc_ne  = (roc_auc_score(yte[non_esc], pred[non_esc])
                   if (non_esc.sum() > 0 and len(np.unique(yte[non_esc])) > 1)
                   else np.nan)
        sensitivity_gate[a]["esc_rate"].append(esc)
        sensitivity_gate[a]["auc_ne"].append(auc_ne)
        sensitivity_gate[a]["cp_cov"].append(cp_cov)

print("\n-- Sensitivity Analysis: KL + CP-Gating only across alpha values --")
print(f"  {'alpha':>6}  {'Esc%':>8}  {'AUC(non-esc)':>12}  {'CPCov':>8}  {'Guarantee':>9}")
print(f"  {'-'*6}  {'-'*8}  {'-'*12}  {'-'*8}  {'-'*9}")
for a in ALPHAS:
    em = np.nanmean(sensitivity_gate[a]["esc_rate"])
    am = np.nanmean(sensitivity_gate[a]["auc_ne"])
    cm = np.nanmean(sensitivity_gate[a]["cp_cov"])
    print(f"  {a:>6.2f}  {em:>7.1f}%  {am:>12.4f}  {cm:>8.3f}  {1-a:>9.2f}")

---
## 📊 Cell 7 — Print Results Table

In [ ]:
def ci(arr):
    a = np.array(arr, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) == 0:
        return np.nan, np.nan, np.nan
    m, s, n = a.mean(), a.std(), len(a)
    return m, m - 1.96*s/np.sqrt(n), m + 1.96*s/np.sqrt(n)

LABEL_MAP = {
    "ml_only":         "LR Only",
    "std_moe":         "Standard MoE",
    "pref_moe":        "Pref MoE (log-barrier)",
    "pref_moe_kl":     "Pref MoE (KL-reg)",
    "conf_kl":         "KL + OOD only",
    "lb_cp_ml":        "LB + CP-ML only",
    "lb_cp_gating":    "LB + CP-gating only",
    "lb_cp_both":      "LB + CP-both",
    "kl_cp_ml":        "KL + CP-ML only",
    "kl_cp_gating":    "KL + CP-gating only",
    "kl_cp_both":      "KL + CP-both",
}

def get_mean(key, metric):
    arr = np.array(records[key][metric], dtype=float)
    arr = arr[~np.isnan(arr)]
    return arr.mean() if len(arr) > 0 else np.nan

# ════════════════════════════════════════════════════════════════════
# Table 1 — Predictive Performance
# ════════════════════════════════════════════════════════════════════
print("\n── Table 1: Predictive Performance ──")
print("=" * 118)
print(f"  {'Model':<38} {'AUC(all)':>8}  {'95% CI':>16}  {'AUC(non-esc)':>12}  "
      f"{'Brier(all)':>10}  {'Brier(ne)':>9}  {'ECE(all)':>8}  {'ECE(ne)':>7}")
print("=" * 118)
for key in KEYS:
    am, al, ah = ci(records[key]["auc"])
    if np.isnan(am):
        print(f"  {LABEL_MAP[key]:<38}  N/A")
        continue
    na  = get_mean(key, "auc_non_abstain")
    bs  = get_mean(key, "brier")
    bsne = get_mean(key, "brier_non_esc")
    na_str   = f"{na:>12.4f}"   if not np.isnan(na)   else "         N/A"
    bs_str   = f"{bs:>10.4f}"   if not np.isnan(bs)   else "       N/A"
    bsne_str = f"{bsne:>14.4f}" if not np.isnan(bsne) else "           N/A"
    ece    = get_mean(key, "ece")
    ece_ne = get_mean(key, "ece_non_esc")
    ece_str    = f"{ece:>8.4f}"    if not np.isnan(ece)    else "     N/A"
    ece_ne_str = f"{ece_ne:>7.4f}" if not np.isnan(ece_ne) else "    N/A"
    print(f"  {LABEL_MAP[key]:<38} {am:>8.4f}  [{al:.4f}-{ah:.4f}]  {na_str}  "
      f"{bs_str}  {bsne_str}  {ece_str}  {ece_ne_str}")
print("=" * 118)

# ════════════════════════════════════════════════════════════════════
# Table 2 — Safety Mechanism Behaviour
# ════════════════════════════════════════════════════════════════════
print("\n── Table 2: Safety Mechanism Behaviour ──")
print("=" * 115)
print(f"  {'Model':<38} {'SoftCov%':>9}  {'EffCov%':>8}  {'Esc%':>6}  {'Abst%':>6}  {'ML Acc':>7}  {'CP Cov':>7}  {'Guarantee':>9}")
print("=" * 115)
for key in KEYS:
    am, *_ = ci(records[key]["auc"])
    if np.isnan(am):
        print(f"  {LABEL_MAP[key]:<38}  N/A")
        continue
    cm,  *_ = ci(records[key]["cov"])
    em,  *_ = ci(records[key]["eff_cov"])
    esm, *_ = ci(records[key]["escalation_rate"])
    abm, *_ = ci(records[key]["abstain_rate"])
    ml   = get_mean(key, "ml_accuracy")
    cc   = get_mean(key, "conformal_coverage")
    es_str = f'{esm:>5.1f}%' if not np.isnan(esm) else '   N/A'
    ab_str   = f"{abm:>5.1f}%"  if (not np.isnan(abm) and abm > 0)  else "   N/A"
    ml_str   = f"{ml:>7.3f}"    if not np.isnan(ml)                  else "    N/A"
    cc_str   = f"{cc:>7.3f}"    if not np.isnan(cc)                  else "    N/A"
    guar_str = f"{(1-ALPHA_CP):>9.2f}" if key in KEYS_WITH_OUTCOMES  else "      N/A"
    eff_str = f"{em:>7.1f}%" if not np.isnan(em) else "    N/A"
    print(f"  {LABEL_MAP[key]:<38} {cm:>8.1f}%  {eff_str}  {es_str}  {ab_str}  {ml_str}  {cc_str}  {guar_str}")
print("=" * 115)
print(f"  Note: CP Cov should be ≥ {1-ALPHA_CP:.2f} (guarantee) for CP models")

# ════════════════════════════════════════════════════════════════════
# Table 3 — Ablation Summary
# ════════════════════════════════════════════════════════════════════
ablation_config = {
    "lb_cp_ml":      (True,  False, False),
    "lb_cp_gating":  (False, True,  False),
    "lb_cp_both":    (True,  True,  False),
    "kl_cp_ml":      (True,  False, False),
    "kl_cp_gating":  (False, True,  False),
    "kl_cp_both":    (True,  True,  False),
    "conf_kl":       (False, False, True),
}

def print_ablation_row(key):
    cp_ml, cp_gate, ood = ablation_config[key]
    esm, *_ = ci(records[key]["escalation_rate"])
    auc_ne   = get_mean(key, "auc_non_abstain")
    cc       = get_mean(key, "conformal_coverage")
    bsne     = get_mean(key, "brier_non_esc")
    es_str   = f"{esm:>5.1f}%"  if (not np.isnan(esm) and esm > 0) else "   N/A"
    auc_str  = f"{auc_ne:>8.4f}" if not np.isnan(auc_ne)            else "     N/A"
    cc_str   = f"{cc:>7.3f}"     if not np.isnan(cc)                else "    N/A"
    bsne_str = f"{bsne:>9.4f}"   if not np.isnan(bsne)              else "      N/A"
    cp_ml_str   = "  ✓  " if cp_ml   else "  —  "
    cp_gate_str = "   ✓    " if cp_gate else "   —    "
    ood_str     = "  ✓  " if ood    else "  —  "
    print(f"  {LABEL_MAP[key]:<38} {cp_ml_str}  {cp_gate_str}  {ood_str}  {es_str}  {auc_str}  {cc_str}  {bsne_str}")

print("\n── Table 3: Ablation Summary (CP models only) ──")
print("=" * 100)
print(f"  {'Model':<38} {'CP-ML':>6}  {'CP-Gate':>8}  {'OOD':>5}  {'Esc%':>6}  {'AUC(ne)':>8}  {'CP Cov':>7}  {'Brier(ne)':>9}")
print("=" * 100)
print("  — Log-Barrier base —")
for key in KEYS_LB:
    print_ablation_row(key)
print("  — KL-reg base —")
for key in KEYS_KL + ["conf_kl"]:
    print_ablation_row(key)
print("=" * 100)

# ════════════════════════════════════════════════════════════════════
# Hyperparameters (Experimental Configuration)
# ════════════════════════════════════════════════════════════════════
print("\n── Experimental Configuration ──")
print(f"  CP alpha (α)        = {ALPHA_CP}     — confidence level for conformal prediction on ML/gating outputs")
print(f"  OOD alpha (α)       = {ALPHA_CONF_OOD}   — confidence level for Mahalanobis OOD detection")
print(f"  KL beta (β)         = {BETA_KL}     — weight of KL divergence term in KL-reg loss")
print(f"  KL mu (μ)           = {MU_COV}     — weight of coverage push term in KL-reg loss")
print(f"  Cal fraction        = {CAL_FRACTION}    — fraction of training fold held out for CP calibration")
print(f"  N seeds × N folds   = {N_SEEDS} × {N_FOLDS}    — total folds = {N_SEEDS * N_FOLDS}")

# ════════════════════════════════════════════════════════════════════
# Outcome Distributions
# ════════════════════════════════════════════════════════════════════
print("\n── Outcome Distributions (% of patients, accumulated across all folds) ──")

outcome_order = ["follow_human", "use_ml", "fallback_human", "escalate", "ood_human", "ood_abstain"]
LABEL_COL = 20  # outcome label column width — must be consistent everywhere
COL = 16        # data column width — must be consistent everywhere

# Header
header = f"  {'Outcome':<{LABEL_COL}}"
for key in KEYS_WITH_OUTCOMES:
    label = LABEL_MAP[key].replace("LB + ", "").replace("KL + ", "").replace(" only", "")
    header += f"  {label:^{COL}}"
print(header)

# Separator
sep = "  " + "-"*LABEL_COL + ("  " + "-"*COL) * len(KEYS_WITH_OUTCOMES)
print(sep)

# Data rows
for outcome in outcome_order:
    row = f"  {outcome:<{LABEL_COL}}"
    has_any = False
    for key in KEYS_WITH_OUTCOMES:
        total = sum(outcome_counts_all[key].values())
        count = outcome_counts_all[key].get(outcome, 0)
        if total > 0 and count > 0:
            pct = 100 * count / total
            cell = f"{pct:.1f}%"
            row += f"  {cell:^{COL}}"
            has_any = True
        else:
            row += f"  {'—':^{COL}}"
    if has_any:
        print(row)

# Totals row
print("  " + "-"*LABEL_COL + ("  " + "-"*COL) * len(KEYS_WITH_OUTCOMES))
totals_row = f"  {'Total patients':<{LABEL_COL}}"
for key in KEYS_WITH_OUTCOMES:
    total = sum(outcome_counts_all[key].values())
    totals_row += f"  {str(total):^{COL}}"
print(totals_row)

# Summary rows
print("  " + "-"*LABEL_COL + ("  " + "-"*COL) * len(KEYS_WITH_OUTCOMES))
row_h = f"  {'Handled (%)':<{LABEL_COL}}"
row_e = f"  {'Escalated (%)':<{LABEL_COL}}"
for key in KEYS_WITH_OUTCOMES:
    total = sum(outcome_counts_all[key].values())
    if total == 0:
        row_h += f"  {'N/A':^{COL}}"
        row_e += f"  {'N/A':^{COL}}"
        continue
    esc_count = (outcome_counts_all[key].get("escalate", 0) +
                 outcome_counts_all[key].get("ood_abstain", 0))
    row_h += f"  {f'{100*(total-esc_count)/total:.1f}%':^{COL}}"
    row_e += f"  {f'{100*esc_count/total:.1f}%':^{COL}}"
print(row_h)
print(row_e)

# ════════════════════════════════════════════════════════════════════
# Best CP Variant
# ════════════════════════════════════════════════════════════════════
print("\n── Best CP Variant ──")
best_key = None
best_auc_ne = -1
best_esc = 100
for key in KEYS_WITH_OUTCOMES:
    auc_ne = get_mean(key, "auc_non_abstain")
    esm, *_ = ci(records[key]["escalation_rate"])
    if not np.isnan(auc_ne) and (auc_ne > best_auc_ne or
        (abs(auc_ne - best_auc_ne) < 0.005 and esm < best_esc)):
        best_auc_ne = auc_ne
        best_esc = esm
        best_key = key
print(f"  {LABEL_MAP[best_key]} — AUC(non-esc): {best_auc_ne:.4f}, Esc: {best_esc:.1f}%")

---
## 📊 Cell 8 — Statistical Signifiance Testing

In [ ]:
from scipy.stats import wilcoxon, ttest_1samp

print('── Statistical Significance Testing (Wilcoxon Signed-Rank) ──')
print('  Baseline: Pref MoE (log-barrier) — original Pradier et al. model')
print('  H0: no difference in means | p < 0.05 = significant')
print('  Innovation B: CP-ML tested (CP-gating inactive on Pima — 0% escalation)\n')

METRICS_A = [
    ('cov',             'Soft Cov%',    True),
    ('auc_non_abstain', 'AUC(non-esc)', True),
    ('brier_non_esc',   'Brier(ne)',    False),
    ('ece_non_esc',     'ECE(ne)',      False),
]

METRICS_B = [
    ('auc_non_abstain', 'AUC(non-esc)', True),
    ('brier_non_esc',   'Brier(ne)',    False),
    ('ece_non_esc',     'ECE(ne)',      False),
    ('ml_accuracy',     'ML Acc',       True),  # Pima only — MIMIC omits this
    # because CP-gating is the operative mechanism on MIMIC (not CP-ML),
    # so ml_accuracy on routed patients is less meaningful there
    ]

col = 32
header  = (f"  {'Model':<{col}} {'Metric':<16} {'Baseline':>9}  "
           f"{'Model':>9}  {'Diff':>8}  {'p-value':>9}  {'Sig':>4}")
divider = ('  ' + '-'*col + '  ' + '-'*15 + '  ' + '-'*9 +
           '  ' + '-'*9 + '  ' + '-'*8 + '  ' + '-'*9 + '  ' + '-'*4)

def run_wilcoxon_row(label, key, baseline_key, metric,
                     metric_label, higher_better, col):
    model_arr    = np.array(records[key][metric],          dtype=float)
    baseline_arr = np.array(records[baseline_key][metric], dtype=float)
    min_len      = min(len(model_arr), len(baseline_arr))
    model_arr    = model_arr[:min_len]
    baseline_arr = baseline_arr[:min_len]
    mask         = ~(np.isnan(model_arr) | np.isnan(baseline_arr))
    model_arr    = model_arr[mask]
    baseline_arr = baseline_arr[mask]
    if len(model_arr) < 10:
        print(f'  {label:<{col}} {metric_label:<16}  insufficient data')
        return
    diff = model_arr.mean() - baseline_arr.mean()
    try:
        _, p = wilcoxon(model_arr, baseline_arr)
    except ValueError:
        print(f'  {label:<{col}} {metric_label:<16}  no difference (p=1.000)')
        return
    sig       = '***' if p < 0.001 else '** ' if p < 0.01 else '*  ' if p < 0.05 else 'n.s.'
    direction = '↑' if (diff > 0 and higher_better) or \
                       (diff < 0 and not higher_better) else '↓'
    print(f'  {label:<{col}} {metric_label:<16} {baseline_arr.mean():>9.4f}  '
          f'{model_arr.mean():>9.4f}  {diff:>+8.4f}  {p:>9.4f}  {sig} {direction}')

# ════════════════════════════════════════════════════════════════════
# Innovation A — KL regularisation vs Log-barrier
# Primary claim: KL-reg improves soft coverage over log-barrier
# Known trade-off: KL may degrade calibration — expected
# ════════════════════════════════════════════════════════════════════
print('  — Innovation A: KL regularisation vs Log-barrier —')
print('  Primary claim: soft coverage improvement')
print('  Known trade-off: Brier/ECE degradation expected from KL objective\n')
print(header)
print(divider)
for metric, metric_label, higher_better in METRICS_A:
    run_wilcoxon_row('Pref MoE (KL-reg)', 'pref_moe_kl', 'pref_moe',
                     metric, metric_label, higher_better, col)
print()

# ════════════════════════════════════════════════════════════════════
# Innovation B — CP-ML safety mechanism
# Note: CP-gating is inactive on Pima (0% escalation) — not tested
# AUC(ne)/Brier(ne) comparisons are on different patient subsets
# after escalation — primary findings are escalation rate + CP Cov
# ════════════════════════════════════════════════════════════════════
print('  — Innovation B: CP-ML safety mechanism —')
print('  Note: CP-gating inactive on Pima (0% escalation) — excluded.')
print('  Metrics computed on different subsets after escalation.')
print('  Primary Innovation B findings are escalation rate and CP coverage.\n')
print(header)
print(divider)

print('  vs Pref LB (does CP-ML add safety value over base LB?):')
for metric, metric_label, higher_better in METRICS_B:
    run_wilcoxon_row('LB + CP-ML only', 'lb_cp_ml', 'pref_moe',
                     metric, metric_label, higher_better, col)
print()

print('  vs Pref KL (does CP-ML add safety value over base KL?):')
for metric, metric_label, higher_better in METRICS_B:
    run_wilcoxon_row('KL + CP-ML only', 'kl_cp_ml', 'pref_moe_kl',
                     metric, metric_label, higher_better, col)
print()

# ════════════════════════════════════════════════════════════════════
# Innovation B — CP coverage vs theoretical guarantee (1-α = 0.90)
# One-sample t-test against fixed threshold — tests whether each
# model statistically satisfies or violates the conformal guarantee
# Note: CP-gating models excluded (inactive — CP Cov = 1.000 tautology)
# ════════════════════════════════════════════════════════════════════
print('  — Innovation B: CP coverage vs theoretical guarantee —')
print(f'  One-sample t-test against guarantee threshold (1-α = {1-ALPHA_CP:.2f})')
print('  Note: CP-gating excluded (inactive on Pima — coverage is tautological)\n')
print(f"  {'Model':<{col}} {'Metric':<16} {'Guarantee':>9}  "
      f"{'Empirical':>9}  {'Diff':>8}  {'p-value':>9}  {'Sig':>4}")
print(divider)

for key, label in [('lb_cp_ml', 'LB + CP-ML only'),
                   ('kl_cp_ml', 'KL + CP-ML only')]:
    cov_arr = np.array(records[key]['conformal_coverage'], dtype=float)
    cov_arr = cov_arr[~np.isnan(cov_arr)]
    if len(cov_arr) < 10:
        print(f'  {label:<{col}}  insufficient data')
        continue
    _, p      = ttest_1samp(cov_arr, 1 - ALPHA_CP)
    sig       = '***' if p < 0.001 else '** ' if p < 0.01 else '*  ' if p < 0.05 else 'n.s.'
    mean      = cov_arr.mean()
    diff      = mean - (1 - ALPHA_CP)
    direction = '↑ satisfies' if diff >= 0 else '↓ violates'
    print(f'  {label:<{col}} {"CP Cov":<16} {(1-ALPHA_CP):>9.3f}  '
          f'{mean:>9.3f}  {diff:>+8.3f}  {p:>9.4f}  {sig} {direction}')
print()

# ════════════════════════════════════════════════════════════════════
# Interaction — does KL-reg reduce CP-ML escalation burden?
# Tests whether Innovation A reduces the need for Innovation B to
# intervene — the core argument for combining both innovations
# ════════════════════════════════════════════════════════════════════
print('  — Interaction: does KL-reg reduce CP-ML escalation burden? —')
print('  Tests whether Innovation A reduces Innovation B intervention rate\n')
print(f"  {'Comparison':<{col}} {'Metric':<16} {'LB':>9}  "
      f"{'KL':>9}  {'Diff':>8}  {'p-value':>9}  {'Sig':>4}")
print(divider)

# Escalation rate
lb_esc = np.array(records['lb_cp_ml']['escalation_rate'], dtype=float)
kl_esc = np.array(records['kl_cp_ml']['escalation_rate'], dtype=float)
min_len = min(len(lb_esc), len(kl_esc))
lb_esc  = lb_esc[:min_len];  kl_esc = kl_esc[:min_len]
mask    = ~(np.isnan(lb_esc) | np.isnan(kl_esc))
lb_esc  = lb_esc[mask];      kl_esc = kl_esc[mask]
try:
    _, p  = wilcoxon(lb_esc, kl_esc)
    sig   = '***' if p < 0.001 else '** ' if p < 0.01 else '*  ' if p < 0.05 else 'n.s.'
    diff  = kl_esc.mean() - lb_esc.mean()
    direction = '↑' if diff < 0 else '↓'
    print(f'  {"KL vs LB CP-ML":<{col}} {"Esc Rate":<16} '
          f'{lb_esc.mean()*100:>9.1f}  {kl_esc.mean()*100:>9.1f}  '
          f'{diff*100:>+8.1f}  {p:>9.4f}  {sig} {direction}')
except ValueError:
    print('  No significant difference in escalation rates')

# ML accuracy
lb_ml = np.array(records['lb_cp_ml']['ml_accuracy'], dtype=float)
kl_ml = np.array(records['kl_cp_ml']['ml_accuracy'], dtype=float)
min_len = min(len(lb_ml), len(kl_ml))
lb_ml   = lb_ml[:min_len];  kl_ml = kl_ml[:min_len]
mask    = ~(np.isnan(lb_ml) | np.isnan(kl_ml))
lb_ml   = lb_ml[mask];      kl_ml = kl_ml[mask]
try:
    _, p  = wilcoxon(lb_ml, kl_ml)
    sig   = '***' if p < 0.001 else '** ' if p < 0.01 else '*  ' if p < 0.05 else 'n.s.'
    diff  = kl_ml.mean() - lb_ml.mean()
    direction = '↑' if diff > 0 else '↓'
    print(f'  {"KL vs LB CP-ML":<{col}} {"ML Acc":<16} '
          f'{lb_ml.mean():>9.3f}  {kl_ml.mean():>9.3f}  '
          f'{diff:>+8.3f}  {p:>9.4f}  {sig} {direction}')
except ValueError:
    print('  No significant difference in ML accuracy')

print()
print(f'  → KL-reg escalation difference vs log-barrier: '
      f'{abs(kl_esc.mean() - lb_esc.mean())*100:.1f} percentage points')
print()
print('  Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. not significant')
print('  Direction:    ↑ improvement  ↓ worse  '
      '(for escalation rate: ↑ = fewer escalations = better)')

---
## 📊 Cell 8a — Noncomformity Score Diagnostics

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Nonconformity Score Diagnostics — Pima Diabetes
# ════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

diag_data = stored_weights.get((0, 0))
if diag_data is None:
    print('ERROR: seed=0, fold=0 not found in stored_weights')
else:
    th_p  = diag_data['th_p']
    th_k  = diag_data['th_k']
    w_p   = diag_data['w_p']
    w_k   = diag_data['w_k']
    bXcal = diag_data['bXcal']
    gcal  = diag_data['gcal']
    ycal  = diag_data['ycal']

    # ── Compute CP-ML scores (LB base) ───────────────────────────────
    cds_ml = ConformalDeferralSystem(alpha_ml=ALPHA_CP,
                                     cp_gating_on=False, cp_ml_on=True)
    cds_ml.calibrate(th_p, w_p, bXcal, gcal, ycal)
    ml_scores = cds_ml.cp_ml.scores
    ml_thresh = cds_ml.cp_ml.threshold

    # ── Compute CP-gating scores (LB base) ───────────────────────────
    cds_gate = ConformalDeferralSystem(alpha_gating=ALPHA_CP,
                                       cp_gating_on=True, cp_ml_on=False)
    cds_gate.calibrate(th_p, w_p, bXcal, gcal, ycal)
    gate_scores = cds_gate.cp_gating.scores
    gate_thresh = cds_gate.cp_gating.threshold

    # ── ML predicted probability distribution ────────────────────────
    ml_probs = sigmoid(bXcal @ th_p)

    # ── Print summary statistics ──────────────────────────────────────
    print('── Nonconformity Score Diagnostics (seed=0, fold=0) ──')
    print(f'\nCP-ML:')
    print(f'  Score range : [{ml_scores.min():.4f}, {ml_scores.max():.4f}]')
    print(f'  Mean score  : {ml_scores.mean():.4f}')
    print(f'  Threshold   : {ml_thresh:.4f}')
    print(f'  Threshold vs range: {"WITHIN range → active" if ml_scores.min() < ml_thresh < ml_scores.max() else "OUTSIDE range → inactive"}')
    print(f'  Scores > threshold: {(ml_scores > ml_thresh).mean()*100:.1f}% of calibration patients')

    print(f'\nCP-Gating:')
    print(f'  Score range : [{gate_scores.min():.4f}, {gate_scores.max():.4f}]')
    print(f'  Mean score  : {gate_scores.mean():.4f}')
    print(f'  Threshold   : {gate_thresh:.4f}')
    print(f'  Threshold vs range: {"WITHIN range → active" if gate_scores.min() < gate_thresh < gate_scores.max() else "OUTSIDE range → inactive"}')
    print(f'  Scores > threshold: {(gate_scores > gate_thresh).mean()*100:.1f}% of calibration patients')

    print(f'\nML Classifier (LB base) prediction distribution:')
    print(f'  Mean predicted prob : {ml_probs.mean():.4f}')
    print(f'  Preds < 0.10        : {(ml_probs < 0.10).mean()*100:.1f}%')
    print(f'  Preds 0.10 - 0.30   : {((ml_probs >= 0.10) & (ml_probs < 0.30)).mean()*100:.1f}%')
    print(f'  Preds 0.30 - 0.70   : {((ml_probs >= 0.30) & (ml_probs < 0.70)).mean()*100:.1f}%')
    print(f'  Preds 0.70 - 0.90   : {((ml_probs >= 0.70) & (ml_probs < 0.90)).mean()*100:.1f}%')
    print(f'  Preds > 0.90        : {(ml_probs > 0.90).mean()*100:.1f}%')
    print(f'  Actual positive rate: {ycal.mean()*100:.1f}%')

    BG, PANEL = '#FFFFFF', '#F8F8F8'

    def style_ax(ax):
        ax.set_facecolor(PANEL)
        ax.spines[:].set_color('#CCCCCC')
        ax.tick_params(colors='#333333', labelsize=8)
        ax.xaxis.label.set_color('#222222')
        ax.yaxis.label.set_color('#222222')
        ax.title.set_color('#111111')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor(BG)

    # ── Panel A: CP-ML nonconformity scores ──────────────────────────
    style_ax(axes[0])
    axes[0].hist(ml_scores, bins=40, color='#5B8DB8', edgecolor='none')
    axes[0].axvline(ml_thresh, color='#C0392B', linewidth=2,
                    label=f'Threshold = {ml_thresh:.4f}')
    axes[0].set_xlabel('Nonconformity score', fontsize=9)
    axes[0].set_ylabel('Count', fontsize=9)
    axes[0].set_title(
        '(A)  CP-ML Nonconformity Scores\n'
        '1 − predicted prob of true label',
        fontsize=9.5, fontweight='bold', pad=6,
    )
    axes[0].legend(fontsize=9)

    # ── Panel B: CP-Gating nonconformity scores ───────────────────────
    style_ax(axes[1])
    axes[1].hist(gate_scores, bins=40, color='#2E9E6B', edgecolor='none')
    axes[1].axvline(gate_thresh, color='#C0392B', linewidth=2,
                    label=f'Threshold = {gate_thresh:.4f}')
    axes[1].set_xlabel('Nonconformity score', fontsize=9)
    axes[1].set_ylabel('Count', fontsize=9)
    axes[1].set_title(
        '(B)  CP-Gating Nonconformity Scores\n'
        'uncertainty in routing decision',
        fontsize=9.5, fontweight='bold', pad=6,
    )
    axes[1].legend(fontsize=9)

    # ── Panel C: ML predicted probability distribution ────────────────
    style_ax(axes[2])
    axes[2].hist(ml_probs, bins=40, color='#5B8DB8', edgecolor='none')
    axes[2].axvline(ml_probs.mean(), color='#C0392B', linewidth=2,
                    ls='--', label=f'Mean = {ml_probs.mean():.4f}')
    axes[2].axvline(ycal.mean(), color='#27AE60', linewidth=2,
                    ls=':', label=f'Actual rate = {ycal.mean():.4f}')
    axes[2].set_xlabel('Predicted probability', fontsize=9)
    axes[2].set_ylabel('Count', fontsize=9)
    axes[2].set_title(
        '(C)  LR Predicted Probability Distribution\n'
        f'{(ml_probs < 0.1).mean()*100:.1f}% of predictions < 0.10',
        fontsize=9.5, fontweight='bold', pad=6,
    )
    axes[2].legend(fontsize=9)

    fig.suptitle(
        'Nonconformity Score Diagnostics — Pima Diabetes  (one representative fold)\n'
        f'CP-ML threshold={ml_thresh:.4f}  |  '
        f'CP-Gating threshold={gate_thresh:.4f}  |  '
        f'α={ALPHA_CP}',
        color='#111111', fontsize=11, fontweight='bold', y=1.01,
    )

    plt.tight_layout()
    plt.savefig('pima_nonconformity_diagnostics.png', dpi=200,
                bbox_inches='tight', facecolor=BG)
    plt.show()
    plt.close(fig)
    print('\n✅ Nonconformity diagnostics saved → pima_nonconformity_diagnostics.png')

---
## 📊 Cell 8b — Calibration Analysis

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Calibration Analysis — Pima Diabetes
# ════════════════════════════════════════════════════════════════════

from scipy.special import expit as sigmoid

n_bins    = 10
bin_edges = np.linspace(0, 1, n_bins + 1)

BG, PANEL = '#FFFFFF', '#F8F8F8'

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.spines[:].set_color('#CCCCCC')
    ax.tick_params(colors='#333333', labelsize=8)
    ax.xaxis.label.set_color('#222222')
    ax.yaxis.label.set_color('#222222')
    ax.title.set_color('#111111')

def reliability_bins(all_preds, all_labels, bin_edges):
    """Compute per-bin mean predicted prob and observed frequency."""
    centres, freqs, counts = [], [], []
    for i in range(len(bin_edges) - 1):
        mask = (all_preds >= bin_edges[i]) & (all_preds < bin_edges[i + 1])
        if mask.sum() > 0:
            centres.append(all_preds[mask].mean())
            freqs.append(all_labels[mask].mean())
            counts.append(mask.sum())
    return np.array(centres), np.array(freqs), np.array(counts)

def collect_preds_pima(key):
    """
    Collect predictions and labels across all folds for a given model key.
    Uses stored_weights which contains w_s (std), w_p (LB), w_k (KL),
    bXval, gval, yval for each (seed, fold).
    """
    preds, labels = [], []
    for data in stored_weights.values():
        bXval = data['bXte']
        gval  = data['gte']
        yval  = data['yte']
        if key == 'ml_only':
            preds.append(sigmoid(bXval @ data['th_ml']))  # was 'th_p'
        elif key == 'pref_moe':
            preds.append(moe_predict(data['th_p'], data['w_p'], None, gval, bXval))
        elif key == 'pref_moe_kl':
            preds.append(moe_predict(data['th_k'], data['w_k'], None, gval, bXval))
        labels.append(yval)
    return np.concatenate(preds), np.concatenate(labels)

# ── Collect predictions for all three models ──────────────────────────────
models = {
    'ML Only':        ('ml_only',      '#5B8DB8'),
    'Pref MoE (LB)':  ('pref_moe',     '#2E9E6B'),
    'Pref MoE (KL)':  ('pref_moe_kl',  '#9B59B6'),
}

# ════════════════════════════════════════════════════════════════════
# Part 1 — Reliability diagrams + prediction distributions (2×3 grid)
# ════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(2, 3, figsize=(18, 10))
fig1.patch.set_facecolor(BG)
fig1.suptitle(
    'Reliability Diagrams — ML Only vs LB vs KL  |  Pima Diabetes\n'
    'Investigating KL calibration degradation vs coverage gain',
    fontsize=13, fontweight='bold', color='#111111',
)

for col, (label, (key, colour)) in enumerate(models.items()):
    all_preds, all_labels = collect_preds_pima(key)
    centres, freqs, counts = reliability_bins(all_preds, all_labels, bin_edges)

    # Top row: reliability diagram
    ax = axes1[0, col]
    style_ax(ax)
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
    ax.scatter(centres, freqs,
               s=counts / counts.max() * 300,
               color=colour, zorder=5, label=label)
    ax.plot(centres, freqs, color=colour, lw=1.5, alpha=0.7)
    ax.fill_between(centres, centres, freqs,
                    alpha=0.15, color=colour, label='Calibration gap')
    ax.set_xlabel('Mean predicted probability', fontsize=9)
    ax.set_ylabel('Observed positive rate', fontsize=9)
    ax.set_title(
        f'{label}\nReliability Diagram',
        fontsize=10, fontweight='bold', pad=6,
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8, facecolor='white', labelcolor='#222222',
              edgecolor='#CCCCCC', framealpha=0.92)

    # Bottom row: prediction distribution
    ax = axes1[1, col]
    style_ax(ax)
    ax.hist(all_preds, bins=40, color=colour, edgecolor='none', alpha=0.8)
    ax.axvline(all_preds.mean(), color='#C0392B', lw=1.8, ls='--',
               label=f'Mean = {all_preds.mean():.3f}')
    ax.axvline(all_labels.mean(), color='#27AE60', lw=1.8, ls=':',
               label=f'Actual rate = {all_labels.mean():.3f}')
    ax.set_xlabel('Predicted probability', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.set_title(f'{label}\nPrediction Distribution', fontsize=10,
                 fontweight='bold', pad=6)
    ax.legend(fontsize=8, facecolor='white', labelcolor='#222222',
              edgecolor='#CCCCCC', framealpha=0.92)

    # Print summary stats
    ece   = np.nanmean(records[key]['ece'])
    brier = np.nanmean(records[key]['brier'])
    print(f'\n  {label}')
    print(f'    ECE:              {ece:.4f}')
    print(f'    Brier:            {brier:.4f}')
    print(f'    Mean pred prob:   {all_preds.mean():.4f}')
    print(f'    Actual pos rate:  {all_labels.mean():.4f}')
    print(f'    Preds > 0.9:      {(all_preds > 0.9).mean()*100:.1f}%')
    print(f'    Preds < 0.1:      {(all_preds < 0.1).mean()*100:.1f}%')

plt.tight_layout()
plt.savefig('pima_calibration_comparison.png', dpi=200,
            bbox_inches='tight', facecolor=BG)
plt.show()
plt.close(fig1)
print('\n✅ Pima calibration comparison saved → pima_calibration_comparison.png')

# ════════════════════════════════════════════════════════════════════
# Part 2 — Single reliability diagram for LB base model
# ════════════════════════════════════════════════════════════════════
all_preds_lb, all_labels_lb = collect_preds_pima('pref_moe')
centres_lb, freqs_lb, counts_lb = reliability_bins(
    all_preds_lb, all_labels_lb, bin_edges
)

fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.patch.set_facecolor(BG)

# Panel A: reliability diagram
style_ax(axes2[0])
axes2[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axes2[0].scatter(
    centres_lb, freqs_lb,
    s=counts_lb / counts_lb.max() * 300,
    color='#2E9E6B', zorder=5, label='LB MoE predictions',
)
axes2[0].plot(centres_lb, freqs_lb, color='#2E9E6B', lw=1.5, alpha=0.7)
axes2[0].fill_between(
    centres_lb, centres_lb, freqs_lb,
    alpha=0.15, color='#CC4444', label='Calibration gap',
)
axes2[0].set_xlabel('Mean predicted probability', fontsize=9)
axes2[0].set_ylabel('Observed positive rate', fontsize=9)
axes2[0].set_title(
    '(A)  Reliability Diagram — Pima Diabetes\n'
    'Log-Barrier base model',
    fontsize=9.5, fontweight='bold', pad=6,
)
axes2[0].set_xlim(0, 1)
axes2[0].set_ylim(0, 1)
axes2[0].legend(fontsize=8, facecolor='white', labelcolor='#222222',
                edgecolor='#CCCCCC', framealpha=0.92)

# Panel B: prediction distribution
style_ax(axes2[1])
axes2[1].hist(all_preds_lb, bins=40, color='#2E9E6B', edgecolor='none')
axes2[1].axvline(all_preds_lb.mean(), color='#C0392B', lw=1.8, ls='--',
                 label=f'Mean predicted = {all_preds_lb.mean():.4f}')
axes2[1].axvline(all_labels_lb.mean(), color='#27AE60', lw=1.8, ls=':',
                 label=f'Actual rate = {all_labels_lb.mean():.4f}')
axes2[1].set_xlabel('Predicted probability', fontsize=9)
axes2[1].set_ylabel('Count', fontsize=9)
axes2[1].set_title(
    '(B)  LB Prediction Distribution — Pima Diabetes\n'
    'distribution of predicted probabilities across all folds',
    fontsize=9.5, fontweight='bold', pad=6,
)
axes2[1].legend(fontsize=8, facecolor='white', labelcolor='#222222',
                edgecolor='#CCCCCC', framealpha=0.92)

fig2.suptitle(
    'LB Base Model Calibration — Pima Diabetes\n'
    f'ECE = {np.nanmean(records["pref_moe"]["ece"]):.4f}  |  '
    f'Brier = {np.nanmean(records["pref_moe"]["brier"]):.4f}  |  25-fold CV',
    color='#111111', fontsize=11, fontweight='bold', y=1.01,
)

plt.tight_layout()
plt.savefig('pima_lb_calibration.png', dpi=200,
            bbox_inches='tight', facecolor=BG)
plt.show()
plt.close(fig2)
print('✅ Pima LB calibration figure saved → pima_lb_calibration.png')

---
## 🔍 Cell 9 — Interpretability: Top Gating Weights

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Gating Weights Analysis — Pima Diabetes
# Shows which features drive the KL vs LB routing decision
# ════════════════════════════════════════════════════════════════════

TOP_N  = 7        # Pima has 8 features — top 7 keeps it readable
BG, PANEL = '#FFFFFF', '#F8F8F8'

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.spines[:].set_color('#CCCCCC')
    ax.tick_params(colors='#333333', labelsize=8)
    ax.xaxis.label.set_color('#222222')
    ax.yaxis.label.set_color('#222222')
    ax.title.set_color('#111111')

# ── Average gating weights across all folds ───────────────────────────────
# stored_weights includes bias term at index 0 (from add_bias),
# so feature weights start at index 1
w_lb_all, w_kl_all = [], []
for data in stored_weights.values():
    w_lb_all.append(data['w_p'][1:])   # drop bias term
    w_kl_all.append(data['w_k'][1:])   # drop bias term

w_lb_mean = np.mean(w_lb_all, axis=0)  # shape: (8,)
w_kl_mean = np.mean(w_kl_all, axis=0)  # shape: (8,)
w_lb_std  = np.std(w_lb_all,  axis=0)
w_kl_std  = np.std(w_kl_all,  axis=0)

# ── Select top N features by absolute KL weight magnitude ─────────────────
sorted_idx   = np.argsort(np.abs(w_kl_mean))[::-1][:TOP_N]
top_features = [feature_names[i] for i in sorted_idx]
top_w_lb     = w_lb_mean[sorted_idx]
top_w_kl     = w_kl_mean[sorted_idx]
top_w_lb_std = w_lb_std[sorted_idx]
top_w_kl_std = w_kl_std[sorted_idx]

# ── Print summary table ───────────────────────────────────────────────────
print(f'\n── Gating Weights: Top {TOP_N} features by |KL weight|  —  Pima Diabetes ──')
print(f'  {"Feature":<35}  {"LB weight":>10}  {"KL weight":>10}  {"Δ (KL-LB)":>10}')
print(f'  {"-"*35}  {"-"*10}  {"-"*10}  {"-"*10}')
for feat, wlb, wkl in zip(top_features, top_w_lb, top_w_kl):
    print(f'  {feat:<35}  {wlb:>10.4f}  {wkl:>10.4f}  {wkl-wlb:>+10.4f}')

# ── Figure: two panels ────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 6))
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(
    1, 2, figure=fig,
    hspace=0.4, wspace=0.38,
    left=0.22, right=0.97,
    top=0.82, bottom=0.08,
)

y_pos = np.arange(TOP_N)
width = 0.35

# ── Panel A: LB vs KL weights side-by-side ────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
style_ax(ax1)

ax1.barh(y_pos - width / 2, top_w_lb, height=width,
         color=COLORS['pref_moe'], edgecolor='none',
         label='Log-Barrier', zorder=3)
ax1.barh(y_pos + width / 2, top_w_kl, height=width,
         color=COLORS['pref_moe_kl'], edgecolor='none',
         label='KL-Reg', zorder=3)

# Error bars (std across folds)
ax1.errorbar(top_w_lb, y_pos - width / 2, xerr=top_w_lb_std,
             fmt='none', color='#333333', capsize=2, lw=0.8, zorder=4)
ax1.errorbar(top_w_kl, y_pos + width / 2, xerr=top_w_kl_std,
             fmt='none', color='#333333', capsize=2, lw=0.8, zorder=4)

ax1.axvline(0, color='#666688', lw=0.8)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_features, fontsize=8, color='#333333')
ax1.invert_yaxis()
ax1.set_xlabel('Gating weight', fontsize=9)
ax1.set_title(
    f'(A)  Gating Weights: LB vs KL\n'
    f'top {TOP_N} features by |KL weight|',
    fontsize=9.5, fontweight='bold', pad=6,
)
ax1.legend(fontsize=8, facecolor='white', labelcolor='#222222',
           edgecolor='#CCCCCC', framealpha=0.92)

# ── Panel B: Delta (KL - LB) weights ──────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
style_ax(ax2)

delta_w    = top_w_kl - top_w_lb
bar_colors = [COLORS['pref_moe_kl'] if d > 0 else COLORS['pref_moe']
              for d in delta_w]

ax2.barh(y_pos, delta_w, height=0.6,
         color=bar_colors, edgecolor='none', zorder=3)
ax2.axvline(0, color='#666688', lw=0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(top_features, fontsize=8, color='#333333')
ax2.invert_yaxis()
ax2.set_xlabel('Δ weight  (KL − LB)', fontsize=9)
ax2.set_title(
    '(B)  Weight Shift: KL − Log-Barrier\n'
    'purple = KL weights more heavily  |  green = LB weights more heavily',
    fontsize=9.5, fontweight='bold', pad=6,
)

fig.suptitle(
    'Gating Weights Analysis — Pima Diabetes\n'
    'Mean gating weights averaged across 25 folds  |  '
    'positive weight = feature increases P(route to ML)',
    color='#111111', fontsize=11, fontweight='bold', y=1.0,
)

plt.savefig('pima_gating_weights.png', dpi=200,
            bbox_inches='tight', facecolor=BG)
plt.show()
plt.close(fig)
print(f'\n✅ Gating weights figure saved → pima_gating_weights.png')